In [29]:
def get_table_info(table_name: str):
    """
    Returns the count of rows and columns for a given table name.
    """    
    # Get row count
    row_count = table_name.count()
    
    # Get column count
    col_count = len(table_name.columns)
    
    return row_count, col_count

In [30]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import broadcast

spark = (SparkSession.builder
         .appName("perfomr-joins")
         .master("spark://spark-master:7077")
         .config("spark.executor.memory", "512m")
         .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")

In [31]:
cards_df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("nullValue", "null")
    .load("../data/Credit Card/CardBase.csv")
)

In [32]:
cards_df.printSchema()

root
 |-- Card_Number: string (nullable = true)
 |-- Card_Family: string (nullable = true)
 |-- Credit_Limit: string (nullable = true)
 |-- Cust_ID: string (nullable = true)



In [33]:
rows, cols = get_table_info(cards_df)
print(f"Table '{cards_df}' has:\n {rows} rows\n {cols} columns.")

Table 'DataFrame[Card_Number: string, Card_Family: string, Credit_Limit: string, Cust_ID: string]' has:
 500 rows
 4 columns.


In [34]:
customers_df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("nullValue", "null")
    .load("../data/Credit Card/CustomerBase.csv")
)

In [35]:
customers_df.printSchema()

root
 |-- Cust_ID: string (nullable = true)
 |-- Age: string (nullable = true)
 |-- Customer_Segment: string (nullable = true)
 |-- Customer_Vintage_Group: string (nullable = true)



In [36]:
rows, cols = get_table_info(customers_df)
print(f"Table '{cards_df}' has:\n {rows} rows\n {cols} columns.")

Table 'DataFrame[Card_Number: string, Card_Family: string, Credit_Limit: string, Cust_ID: string]' has:
 5674 rows
 4 columns.


In [37]:
transactions_df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("nullValue", "null")
    .load("../data/Credit Card/TransactionBase.csv")
)

In [38]:
transactions_df.printSchema()

root
 |-- Transaction_ID: string (nullable = true)
 |-- Transaction_Date: string (nullable = true)
 |-- Credit_Card_ID: string (nullable = true)
 |-- Transaction_Value: string (nullable = true)
 |-- Transaction_Segment: string (nullable = true)



In [39]:
rows, cols = get_table_info(transactions_df)
print(f"Table '{cards_df}' has:\n {rows} rows\n {cols} columns.")

Table 'DataFrame[Card_Number: string, Card_Family: string, Credit_Limit: string, Cust_ID: string]' has:
 10000 rows
 5 columns.


In [40]:
fraud_df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("nullValue", "null")
    .load("../data/Credit Card/FraudBase.csv")
)

In [41]:
fraud_df.printSchema()

root
 |-- Transaction_ID: string (nullable = true)
 |-- Fraud_Flag: string (nullable = true)



In [42]:
rows, cols = get_table_info(fraud_df)
print(f"Table '{cards_df}' has:\n {rows} rows\n {cols} columns.")

Table 'DataFrame[Card_Number: string, Card_Family: string, Credit_Limit: string, Cust_ID: string]' has:
 109 rows
 2 columns.


In [43]:
# inner
customer_cards_df = (
    customers_df.join(
         cards_df
        ,on="Cust_ID"
        ,how="inner"
    )
)

customer_cards_df.show()

+-------+---+----------------+----------------------+-------------------+-----------+------------+
|Cust_ID|Age|Customer_Segment|Customer_Vintage_Group|        Card_Number|Card_Family|Credit_Limit|
+-------+---+----------------+----------------------+-------------------+-----------+------------+
|CC55858| 30|         Diamond|                   VG1|2868-5606-5152-5706|       Gold|       27000|
|CC46077| 49|         Diamond|                   VG1|6876-7378-4945-3251|       Gold|       44000|
|CC46484| 49|         Diamond|                   VG1|5556-4557-4566-1540|       Gold|       45000|
|CC59340| 25|         Diamond|                   VG1|5618-9718-9367-2102|       Gold|       14000|
|CC62994| 48|         Diamond|                   VG1|1652-7516-1273-1992|   Platinum|      180000|
|CC43841| 30|         Diamond|                   VG1|7212-8665-7734-5918|   Platinum|       55000|
|CC21312| 45|         Diamond|                   VG1|7837-4036-5999-1672|       Gold|       24000|
|CC90510| 

In [44]:
customer_cards_df.count()

500

In [45]:
joined_transactions_df = (
    transactions_df.join(
        fraud_df
        ,on="Transaction_ID"
        ,how="left_outer"
    )
)

In [46]:
joined_transactions_df.show()

+--------------+----------------+-------------------+-----------------+-------------------+----------+
|Transaction_ID|Transaction_Date|     Credit_Card_ID|Transaction_Value|Transaction_Segment|Fraud_Flag|
+--------------+----------------+-------------------+-----------------+-------------------+----------+
|  CTID28830551|       24-Apr-16|1629-9566-3285-2123|            23649|              SEG25|      null|
|  CTID45504917|       11-Feb-16|3697-6001-4909-5350|            26726|              SEG16|      null|
|  CTID47312290|        1-Nov-16|5864-4475-3659-1440|            22012|              SEG14|      null|
|  CTID25637718|       28-Jan-16|5991-4421-8476-3804|            37637|              SEG17|      null|
|  CTID66743960|       17-Mar-16|1893-8853-9900-8478|             5113|              SEG14|      null|
|  CTID22308010|       15-May-16|5206-5979-9383-4538|             9551|              SEG13|      null|
|  CTID41917614|       11-Jul-16|5129-6974-6371-2964|            29511|  

In [47]:
joined_transactions_df.count()

10000

In [48]:
joinExpr = (
    (
        (customer_cards_df["Card_Number"] == joined_transactions_df["Credit_Card_ID"])
            &
        (joined_transactions_df["Fraud_Flag"].isNotNull())
    )
)

customer_with_fraud_df = (
    customer_cards_df.join(
         joined_transactions_df
        ,on =joinExpr
        ,how="inner"
    )
)

customer_with_fraud_df.show(2)

+-------+---+----------------+----------------------+-------------------+-----------+------------+--------------+----------------+-------------------+-----------------+-------------------+----------+
|Cust_ID|Age|Customer_Segment|Customer_Vintage_Group|        Card_Number|Card_Family|Credit_Limit|Transaction_ID|Transaction_Date|     Credit_Card_ID|Transaction_Value|Transaction_Segment|Fraud_Flag|
+-------+---+----------------+----------------------+-------------------+-----------+------------+--------------+----------------+-------------------+-----------------+-------------------+----------+
|CC87306| 30|         Diamond|                   VG1|5734-5619-8469-4044|       Gold|       36000|  CTID26555772|       11-Jan-16|5734-5619-8469-4044|              683|              SEG22|         1|
|CC87034| 36|        Platinum|                   VG2|6722-7299-6082-7974|       Gold|       34000|  CTID30763806|       17-Dec-16|6722-7299-6082-7974|            40751|              SEG21|         1|


### Right Outer Join

In [49]:
 data1 = [("Alice", "F", 25), ("Bob", "M", 30), ("Charlie", "M", 35), ("Dave", "F", 40)]
df1 = spark.createDataFrame(data1, ["Name", "Gender", "Age"])

data2 = [("Charlie", "M"), ("Dave", "M"), ("Eve", "F")]
df2 = spark.createDataFrame(data2, ["Name", "Gender"])

In [50]:
right_join = df1.join(df2, on='Name', how='right_outer')
right_join.show()

+-------+------+----+------+
|   Name|Gender| Age|Gender|
+-------+------+----+------+
|   Dave|     F|  40|     M|
|    Eve|  null|null|     F|
|Charlie|     M|  35|     M|
+-------+------+----+------+



### Full outer join

In [51]:
full_join = df1.join(df2, on='Name', how='outer')
full_join.show()

+-------+------+----+------+
|   Name|Gender| Age|Gender|
+-------+------+----+------+
|  Alice|     F|  25|  null|
|    Bob|     M|  30|  null|
|Charlie|     M|  35|     M|
|   Dave|     F|  40|     M|
|    Eve|  null|null|     F|
+-------+------+----+------+



### Cross join

In [52]:
cross_join = df1.crossJoin(df2)
cross_join.show()

+-------+------+---+-------+------+
|   Name|Gender|Age|   Name|Gender|
+-------+------+---+-------+------+
|  Alice|     F| 25|Charlie|     M|
|    Bob|     M| 30|Charlie|     M|
|  Alice|     F| 25|   Dave|     M|
|  Alice|     F| 25|    Eve|     F|
|    Bob|     M| 30|   Dave|     M|
|    Bob|     M| 30|    Eve|     F|
|Charlie|     M| 35|Charlie|     M|
|   Dave|     F| 40|Charlie|     M|
|Charlie|     M| 35|   Dave|     M|
|Charlie|     M| 35|    Eve|     F|
|   Dave|     F| 40|   Dave|     M|
|   Dave|     F| 40|    Eve|     F|
+-------+------+---+-------+------+



### Broadcast join

In [53]:
broadcast_join = df1.join(broadcast(df2), ["Name", "Gender"], "inner")
broadcast_join.show()

+-------+------+---+
|   Name|Gender|Age|
+-------+------+---+
|Charlie|     M| 35|
+-------+------+---+



### Multiple Join Conditions

In [54]:
multi_join = df1.join(df2, on=['Name', 'Gender'], how='inner')
multi_join.show()

+-------+------+---+
|   Name|Gender|Age|
+-------+------+---+
|Charlie|     M| 35|
+-------+------+---+



In [55]:
spark.stop()